# Procesamiento de Matrículas - Cámaras de Tráfico

Este notebook procesa los archivos CSV mensuales de las cámaras de reconocimiento automático de matrículas (ANPR) y genera un archivo consolidado limpio.

## 1. Importar librerías necesarias

In [11]:
import os
import glob
import pandas as pd
from pathlib import Path
from datetime import datetime

print("✓ Librerías importadas correctamente")

✓ Librerías importadas correctamente


## 2. Configurar rutas y buscar archivos CSV

In [12]:
# Definir rutas
carpeta_csv = Path("CSV")
archivo_salida = Path("matriculas_consolidado.csv")

# Buscar todos los archivos CSV en la carpeta
archivos_csv = glob.glob(str(carpeta_csv / "*.csv"))

print(f"📁 Archivos CSV encontrados: {len(archivos_csv)}")
print("="*50)
for i, archivo in enumerate(archivos_csv, 1):
    nombre = os.path.basename(archivo)
    tamaño = os.path.getsize(archivo) / (1024*1024)  # Tamaño en MB
    print(f"{i}. {nombre}")
    print(f"   Tamaño: {tamaño:.2f} MB")

📁 Archivos CSV encontrados: 8
1. 2504 Matriculas Abr 25.csv
   Tamaño: 114.63 MB
2. 2505 Matriculas May 25.csv
   Tamaño: 158.75 MB
3. 2506 Matriculas Jun 25.csv
   Tamaño: 195.81 MB
4. 2507 Matriculas Jul 25.csv
   Tamaño: 243.73 MB
5. 2508 Matriculas Agos 25.csv
   Tamaño: 280.78 MB
6. 2509 Matriculas Sept 25.csv
   Tamaño: 196.00 MB
7. 2510 Matriculas Oct 25.csv
   Tamaño: 157.60 MB
8. 2511 Matriculas Nov 25.csv
   Tamaño: 124.63 MB


## 3. Procesar y limpiar cada archivo CSV

Cada archivo tiene 5 filas informativas al inicio que debemos eliminar:
- Fila 1: Título del reporte
- Fila 2: Sistema
- Fila 3: Hora de exportación
- Fila 4: Número total de registros
- Fila 5: Fila vacía
- Fila 6: **Encabezados de columnas** (esta es la que queremos mantener)

In [13]:
# Lista para almacenar todos los dataframes
dataframes = []

# Procesar cada archivo
for archivo in archivos_csv:
    nombre_archivo = os.path.basename(archivo)
    print(f"\n🔄 Procesando: {nombre_archivo}")
    
    try:
        # Leer el CSV normalmente
        df = pd.read_csv(
            archivo, 
            skiprows=5,  # Saltar las 5 primeras filas informativas
            sep=';',
            encoding='utf-8',
            low_memory=False,
            on_bad_lines='skip'
        )
        
        # Limpiar nombres de columnas
        df.columns = df.columns.str.strip()
        
        print(f"  📋 Columnas originales: {len(df.columns)}")
        
        # SOLUCIÓN: Renombrar columnas moviéndolas una posición a la izquierda
        # Eliminar "Placa de licencia" y renombrar el resto
        nuevos_nombres = list(df.columns[1:])  # Tomar desde la 2da columna en adelante
        df = df.iloc[:, :-1]  # Eliminar la última columna (sobrante)
        df.columns = nuevos_nombres
        
        print(f"  ✂️  Eliminada columna 'Placa de licencia'")
        
        # Eliminar columnas innecesarias
        columnas_a_eliminar = [
            'Lista de vehículos',
            'Propietario',
            'Phone',
            'Velocidad de conducción',
            'Miniaturas'
        ]
        
        df = df.drop(columns=columnas_a_eliminar, errors='ignore')
        
        print(f"  ✂️  Eliminadas columnas innecesarias: {columnas_a_eliminar}")
        print(f"  ✅ Columnas finales: {len(df.columns)}")
        print(f"  📝 Columnas: {list(df.columns)}")
        print(f"  ✅ Procesadas {len(df):,} filas")
        
        # Agregar a la lista
        dataframes.append(df)
        
    except Exception as e:
        print(f"  ❌ Error: {e}")
        import traceback
        traceback.print_exc()

print(f"\n{'='*50}")
print(f"Total de archivos procesados: {len(dataframes)}")


🔄 Procesando: 2504 Matriculas Abr 25.csv
  📋 Columnas originales: 14
  ✂️  Eliminada columna 'Placa de licencia'
  📋 Columnas originales: 14
  ✂️  Eliminada columna 'Placa de licencia'
  ✂️  Eliminadas columnas innecesarias: ['Lista de vehículos', 'Propietario', 'Phone', 'Velocidad de conducción', 'Miniaturas']
  ✅ Columnas finales: 8
  📝 Columnas: ['Número de matrícula', 'Hora', 'Cámara', 'País/Región', 'Tipo de vehículo', 'Marca', 'Color', 'Dirección de conducción']
  ✅ Procesadas 836,056 filas

🔄 Procesando: 2505 Matriculas May 25.csv
  ✂️  Eliminadas columnas innecesarias: ['Lista de vehículos', 'Propietario', 'Phone', 'Velocidad de conducción', 'Miniaturas']
  ✅ Columnas finales: 8
  📝 Columnas: ['Número de matrícula', 'Hora', 'Cámara', 'País/Región', 'Tipo de vehículo', 'Marca', 'Color', 'Dirección de conducción']
  ✅ Procesadas 836,056 filas

🔄 Procesando: 2505 Matriculas May 25.csv
  📋 Columnas originales: 14
  ✂️  Eliminada columna 'Placa de licencia'
  📋 Columnas originales:

## 4. Consolidar todos los datos

In [14]:
if dataframes:
    print("🔗 Consolidando todos los datos...")
    df_consolidado = pd.concat(dataframes, ignore_index=True)
    
    print(f"\n{'='*60}")
    print(f"📊 RESUMEN DEL ARCHIVO CONSOLIDADO")
    print(f"{'='*60}")
    print(f"Total de registros: {len(df_consolidado):,}")
    print(f"Total de columnas: {len(df_consolidado.columns)}")
    print(f"\nColumnas disponibles:")
    for i, col in enumerate(df_consolidado.columns, 1):
        print(f"  {i}. {col}")
else:
    print("❌ No se pudo procesar ningún archivo")

🔗 Consolidando todos los datos...

📊 RESUMEN DEL ARCHIVO CONSOLIDADO
Total de registros: 10,440,449
Total de columnas: 8

Columnas disponibles:
  1. Número de matrícula
  2. Hora
  3. Cámara
  4. País/Región
  5. Tipo de vehículo
  6. Marca
  7. Color
  8. Dirección de conducción

📊 RESUMEN DEL ARCHIVO CONSOLIDADO
Total de registros: 10,440,449
Total de columnas: 8

Columnas disponibles:
  1. Número de matrícula
  2. Hora
  3. Cámara
  4. País/Región
  5. Tipo de vehículo
  6. Marca
  7. Color
  8. Dirección de conducción


## 5. Vista previa de los datos

In [15]:
# Mostrar las primeras 30 filas del dataframe consolidado
df_consolidado.head(30)

,Número de matrícula,Hora,Cámara,País/Región,Tipo de vehículo,Marca,Color,Dirección de conducción
0,"=""Unknown""",2025/04/30 23:59:55,Estibaliz LPR,No compatible,SUV/MPV,Audi,White,Invertir
1,"=""7193KTG""",2025/04/30 23:59:54,Ratlla Del Terme - CV-140 LPR,España,SUV/MPV,Infiniti,Gris,Invertir
2,"=""9965CCT""",2025/04/30 23:59:53,Rotonda La Volta Oeste LPR,España,Coche de salón,Citroen,White,Avance
3,"=""Unknown""",2025/04/30 23:59:51,Estibaliz LPR,No compatible,Coche de salón,Mazda,Negro,Invertir
4,"=""2820BBF""",2025/04/30 23:59:48,Avda. Papa Luna - C. Madrid LPR,Gran Bretaña,Coche de salón,Volkswagen,Gris,Invertir
5,"=""4362FPX""",2025/04/30 23:59:30,Ratlla Del Terme - CV-140 LPR,España,Coche de salón,Peugeot,Gris,Invertir
6,"=""287H7H""",2025/04/30 23:59:27,Avda. Papa Luna - Benicarló LPR,Armenia,SUV/MPV,Porsche,White,Invertir
7,"=""Unknown""",2025/04/30 23:59:22,Ratlla Del Terme - CV-140 LPR,No compatible,Coche de salón,Volkswagen,White,Avance
8,"=""Unknown""",2025/04/30 23:59:18,Rotonda Abellers - Avda.Estacion LPR,No compatible,SUV/MPV,Otros,Verde,Avance
9,"=""MGGD272""",2025/04/30 23:59:12,Estibaliz LPR,Suiza,SUV/MPV,Renault,White,Avance


In [16]:
# Diccionario con las coordenadas de cada cámara
coordenadas_camaras = {
    'Acceso Casco Antiguo - Calle M.Roca LPR': (40.357475, 0.405873),
    'Assegador De La Creu LPR': (40.376017, 0.396905),
    'Avda. Papa Luna - Benicarló LPR': (40.399806, 0.417028),
    'Avda. Papa Luna - C. Madrid LPR': (40.370660, 0.404496),
    'Calle Irta - San Antonio LPR': (40.353433, 0.395024),
    'Estibaliz LPR': (40.367630, 0.393503),
    'Ratlla Del Terme - CV-140 LPR': (40.400784, 0.414709),
    'Rotonda Abellers - Avda.Estacion LPR': (40.378614, 0.387877),
    'Rotonda La Volta Oeste LPR': (40.393551, 0.408298)
}

# Ver todas las cámaras únicas disponibles (solo las que terminan en LPR)
print("📹 LISTADO DE CÁMARAS CON COORDENADAS:")
print("="*100)

camaras_unicas = sorted([c for c in df_consolidado['Cámara'].unique() if str(c).endswith('LPR')])

for i, camara in enumerate(camaras_unicas, 1):
    coords = coordenadas_camaras.get(camara, ('N/A', 'N/A'))
    print(f"{i:2}. {camara:50} - Coords: {coords[0]}, {coords[1]}")

print("="*100)
print(f"Total de cámaras LPR: {len(camaras_unicas)}")

📹 LISTADO DE CÁMARAS CON COORDENADAS:
 1. Acceso Casco Antiguo - Calle M.Roca LPR            - Coords: 40.357475, 0.405873
 2. Assegador De La Creu LPR                           - Coords: 40.376017, 0.396905
 3. Avda. Papa Luna - Benicarló LPR                    - Coords: 40.399806, 0.417028
 4. Avda. Papa Luna - C. Madrid LPR                    - Coords: 40.37066, 0.404496
 5. Calle Irta - San Antonio LPR                       - Coords: 40.353433, 0.395024
 6. Estibaliz LPR                                      - Coords: 40.36763, 0.393503
 7. Ratlla Del Terme - CV-140 LPR                      - Coords: 40.400784, 0.414709
 8. Rotonda Abellers - Avda.Estacion LPR               - Coords: 40.378614, 0.387877
 9. Rotonda La Volta Oeste LPR                         - Coords: 40.393551, 0.408298
Total de cámaras LPR: 9
 1. Acceso Casco Antiguo - Calle M.Roca LPR            - Coords: 40.357475, 0.405873
 2. Assegador De La Creu LPR                           - Coords: 40.376017, 0.396905
 3. A

## 6b. Eliminar duplicados (matrículas únicas)

In [17]:
# Eliminar duplicados: mantener solo la PRIMERA aparición de cada matrícula POR DÍA
# Esto asegura que si un coche pasa por varias cámaras el mismo día, se cuente solo una vez

print("\n" + "="*80)
print("🔍 ELIMINANDO DUPLICADOS (MATRÍCULAS ÚNICAS POR DÍA)")
print("="*80)

# Convertir la columna Hora a datetime y extraer solo la fecha (día)
df_consolidado['Fecha'] = pd.to_datetime(df_consolidado['Hora'], format='%Y/%m/%d %H:%M:%S', errors='coerce').dt.date

# Eliminar duplicados por matrícula + fecha (mismo día)
registros_antes = len(df_consolidado)
df_consolidado_unico = df_consolidado.drop_duplicates(subset=['Número de matrícula', 'Fecha'], keep='first')
registros_despues = len(df_consolidado_unico)
eliminados = registros_antes - registros_despues

print(f"\n📊 Resultado:")
print(f"  Antes: {registros_antes:,} registros")
print(f"  Después: {registros_despues:,} matrículas únicas por día")
print(f"  Eliminados: {eliminados:,} duplicados ({(eliminados/registros_antes*100):.1f}%)")
print("="*80)

print("\n💡 Nota: Cada matrícula se cuenta solo UNA VEZ por día, aunque pase por varias cámaras.")


🔍 ELIMINANDO DUPLICADOS (MATRÍCULAS ÚNICAS POR DÍA)

📊 Resultado:
  Antes: 10,440,449 registros
  Después: 3,636,703 matrículas únicas por día
  Eliminados: 6,803,746 duplicados (65.2%)

💡 Nota: Cada matrícula se cuenta solo UNA VEZ por día, aunque pase por varias cámaras.

📊 Resultado:
  Antes: 10,440,449 registros
  Después: 3,636,703 matrículas únicas por día
  Eliminados: 6,803,746 duplicados (65.2%)

💡 Nota: Cada matrícula se cuenta solo UNA VEZ por día, aunque pase por varias cámaras.


## 7. Guardar archivo CSV consolidado

In [18]:
# Agregar las coordenadas al dataframe
print("📍 Agregando coordenadas a cada registro...")

# Crear columnas Latitud y Longitud mapeando desde el diccionario
df_consolidado_unico['Latitud'] = df_consolidado_unico['Cámara'].map(lambda x: coordenadas_camaras.get(x, (None, None))[0])
df_consolidado_unico['Longitud'] = df_consolidado_unico['Cámara'].map(lambda x: coordenadas_camaras.get(x, (None, None))[1])

# Verificar cuántos registros tienen coordenadas
con_coords = df_consolidado_unico['Latitud'].notna().sum()
sin_coords = df_consolidado_unico['Latitud'].isna().sum()

print(f"✅ Registros con coordenadas: {con_coords:,}")
print(f"⚠️  Registros sin coordenadas: {sin_coords:,}")

if sin_coords > 0:
    camaras_sin_coords = df_consolidado_unico[df_consolidado_unico['Latitud'].isna()]['Cámara'].unique()
    print(f"\n⚠️  Cámaras sin coordenadas definidas:")
    for cam in camaras_sin_coords:
        print(f"   - {cam}")
        
print("\n✅ Coordenadas agregadas correctamente")

📍 Agregando coordenadas a cada registro...


C:\Users\touri\AppData\Local\Temp\ipykernel_3496\2825122116.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_consolidado_unico['Latitud'] = df_consolidado_unico['Cámara'].map(lambda x: coordenadas_camaras.get(x, (None, None))[0])


✅ Registros con coordenadas: 3,630,634
⚠️  Registros sin coordenadas: 6,069

⚠️  Cámaras sin coordenadas definidas:
   - Avda. Papa Luna - Calle Madrid
   - Assegador De La Creu

✅ Coordenadas agregadas correctamente


C:\Users\touri\AppData\Local\Temp\ipykernel_3496\2825122116.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_consolidado_unico['Longitud'] = df_consolidado_unico['Cámara'].map(lambda x: coordenadas_camaras.get(x, (None, None))[1])


In [19]:
# Exportar archivo CSV con todas las cámaras (matrículas únicas por día) incluyendo coordenadas
archivo_salida = Path("matriculas_consolidado.csv")
df_consolidado_unico.to_csv(archivo_salida, index=False, sep=';', encoding='utf-8-sig')
tamaño = os.path.getsize(archivo_salida) / (1024*1024)
df_consolidado_unico.head(40)
print("="*80)
print("✅ ARCHIVO CONSOLIDADO GENERADO")
print("="*80)
print(f"📄 Nombre: {archivo_salida}")
print(f"💾 Tamaño: {tamaño:.2f} MB")
print(f"📊 Matrículas únicas por día: {len(df_consolidado_unico):,}")
print(f"📹 Cámaras incluidas: {len(df_consolidado_unico['Cámara'].unique())}")
print(f"📍 Incluye coordenadas: Latitud y Longitud")
print(f"📅 Fecha de creación: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*80)

✅ ARCHIVO CONSOLIDADO GENERADO
📄 Nombre: matriculas_consolidado.csv
💾 Tamaño: 469.13 MB
📊 Matrículas únicas por día: 3,636,703
📹 Cámaras incluidas: 11
📍 Incluye coordenadas: Latitud y Longitud
📅 Fecha de creación: 2025-12-04 16:57:19
📹 Cámaras incluidas: 11
📍 Incluye coordenadas: Latitud y Longitud
📅 Fecha de creación: 2025-12-04 16:57:19


In [20]:
df_consolidado_unico.tail(40)

,Número de matrícula,Hora,Cámara,País/Región,Tipo de vehículo,Marca,Color,Dirección de conducción,Fecha,Latitud,Longitud
10440238,"=""3281KEM""",2025/11/01 00:19:44,Estibaliz LPR,España,SUV/MPV,Volkswagen,Negro,Avance,2025-11-01,40.367630,0.393503
10440241,"=""9398LCF""",2025/11/01 00:19:15,Estibaliz LPR,España,SUV/MPV,Chery,Rojo,Avance,2025-11-01,40.367630,0.393503
10440266,"=""L340LT""",2025/11/01 00:16:34,Estibaliz LPR,Finlandia,SUV/MPV,Hyundai,White,Avance,2025-11-01,40.367630,0.393503
10440281,"=""E025GBW""",2025/11/01 00:15:03,Estibaliz LPR,Kazajistán,Coche de salón,Subaru,Rojo,Invertir,2025-11-01,40.367630,0.393503
10440293,"=""5318KWP""",2025/11/01 00:13:55,Avda. Papa Luna - Benicarló LPR,España,Coche de salón,Roewe,Negro,Invertir,2025-11-01,40.399806,0.417028
10440310,"=""1459GWT""",2025/11/01 00:11:44,Ratlla Del Terme - CV-140 LPR,España,SUV/MPV,Toyota,Gris,Invertir,2025-11-01,40.400784,0.414709
10440311,"=""CV184LH""",2025/11/01 00:11:36,Ratlla Del Terme - CV-140 LPR,Georgia,Camioneta pickup,Hyundai,Negro,Avance,2025-11-01,40.400784,0.414709
10440314,"=""4476FNP""",2025/11/01 00:11:26,Calle Irta - San Antonio LPR,España,Coche de salón,Seat,Gris,Avance,2025-11-01,40.353433,0.395024
10440317,"=""9351KMN""",2025/11/01 00:11:11,Ratlla Del Terme - CV-140 LPR,España,SUV/MPV,KIA,Negro,Invertir,2025-11-01,40.400784,0.414709
10440318,"=""1761JRY""",2025/11/01 00:11:04,Rotonda Abellers - Avda.Estacion LPR,Mongolia,SUV/MPV,Nissan,Verde,Avance,2025-11-01,40.378614,0.387877
